# 🧬 SPS Self-Specialization — Dynamic Serialization + Capability Handler Demo

**Research claim:** the system starts with only `IntegerMultiplication [S0]`. When `FloatMultiplication` is missing, it dynamically serializes/generalizes the existing State 0 capability, creates `SerializeCapability [S0]`, reparents the existing integer capability without changing its ID, creates a transient `S0-C` copy, specializes the copy with Ollama/Qwen, verifies it, activates the result as `S1`, persists it, reloads it, and reuses it.

## 🔬 Complete research flow

```text
INITIAL
IntegerMultiplication [S0]
        │
        │ request multiply(2.5, 4.0)
        ▼
FloatMultiplication missing
        │
        ▼
SERIALIZE / GENERALIZE existing IntegerMultiplication [S0]
        │
        ▼
SerializeCapability [S0]  ← CREATED NOW
        │
        ├── IntegerMultiplication [S0]  ← SAME ID, reparented
        │
        ▼
REPLICATE → transient copy [S0-C]
        ↓
SPECIALIZE → Ollama + Qwen Coder
        ↓
GENERATED → FloatMultiplication
        ↓
VERIFY → syntax/policy + functional cases
        ↓
ACTIVATE → FloatMultiplication [S1]
        ↓
FINAL
              SerializeCapability [S0]
                        │
               ┌────────┴────────┐
               ▼                 ▼
 IntegerMultiplication    FloatMultiplication
        [S0]                    [S1]

PERSIST → RELOAD → REUSE without another AI call
```

**Important:** `SerializeCapability` does not exist before the float request, and creating it does not change the integer capability from S0 to S1.

## 1. Install and start Ollama first

**Run this cell first.** Ollama is started from `/content`, a stable directory that is not deleted when the repository is recloned.

In [15]:
%cd /content
!apt-get update -qq
!apt-get install -y -qq zstd curl
!curl -fsSL https://ollama.com/install.sh | sh
!pkill -9 ollama || true
!pkill -9 llama-server || true
!nohup ollama serve >/tmp/ollama.log 2>&1 &
!sleep 5
!ollama --version
!curl -sf http://127.0.0.1:11434/api/tags || (cat /tmp/ollama.log; exit 1)
!ollama pull qwen2.5-coder:7b

/content
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
ollama version is 0.33.3
{"models":[{"name":"qwen2.5-coder:7b","model":"qwen2.5-coder:7b","modified_at":"2026-09-04T10:30:01.396538641Z","size":4683087561,"digest":"dae161e27b0e90dd1856c8bb3209201fd6736d8eb66298e75ed87571486f4364","details":{"parent_model":"","format":"gguf","family":"qwen2","families":["qwen2"],"parameter_size":"7.6B","quantization_l

## 2. Clone the latest `main` branch and install dependencies

In [16]:
%cd /content
!rm -rf self-specialization
!git clone --branch main --single-branch -q https://github.com/muhammadnaumantahir/self-specialization.git
%cd /content/self-specialization
!pip -q install -r requirements.txt pytest
!echo 'Repository commit:'
!git rev-parse HEAD
!echo '\nAvailable Ollama models:'
!ollama list

/content
/content/self-specialization
Repository commit:
fe464db9b4a4a3cc8d02f257c48394a153684e75
\nAvailable Ollama models:
NAME                ID              SIZE      MODIFIED      
qwen2.5-coder:7b    dae161e27b0e    4.7 GB    4 seconds ago    


## 3. Run deterministic tests before the real model

These tests validate dynamic serialization, reparenting, S0-C replication, specialization, verification, the final hierarchy, persistence and reload. They do not require Ollama.

In [17]:
%cd /content/self-specialization
!PYTHONPATH=. pytest -q

/content/self-specialization
......................                                                   [100%]
22 passed in 0.62s


## 4. Run the real SPS experiment

The demo creates a fresh registry. Its initial state contains only `IntegerMultiplication [S0]`.

In [24]:
%cd /content/self-specialization
import os
os.environ['OLLAMA_MODEL'] = 'qwen2.5-coder:7b'
!PYTHONPATH=. python experiments/self_specialization_demo.py

/content/self-specialization
🧬 SPS SELF-SPECIALIZATION — MINIMAL RESEARCH PROTOTYPE
Complete research flow:
  INTEGER S0 → MISSING FLOAT REQUEST → SERIALIZE/GENERALIZE → S0
      → REPLICATE → S0-C → SPECIALIZE WITH OLLAMA/QWEN
      → VERIFY → FLOAT S1 → LINK UNDER SERIALIZE → PERSIST → REUSE

Persistent capability registry: /tmp/sps-capability-registry

🔵 PHASE 1 — INITIAL STATE
Capability : IntegerMultiplication [S0]
Contract   : ['int', 'int'] -> int
Test       : 6 × 7 = 42
Parent     : None
Hierarchy  : IntegerMultiplication [S0]
Decision   : No SerializeCapability exists yet.

🟡 PHASE 2 — NEW REQUEST: FLOAT MULTIPLICATION
User request: multiply(2.5, 4.0)
Required    : [float, float] -> float
Lookup      : FloatMultiplication is missing.

The handler/evolution flow is:
  1. Detect that FloatMultiplication does not exist
  2. Serialize/generalize the existing IntegerMultiplication [S0]
  3. Create SerializeCapability [S0] at runtime
  4. Reparent the existing IntegerMultiplication 

## 5. What the demo must prove

### Initial state
```text
IntegerMultiplication [S0]
```
No `SerializeCapability`, `FloatMultiplication`, or permanent replication copy exists.

### Runtime evolution
1. Detect missing `[float, float] -> float`.
2. Select existing `IntegerMultiplication [S0]`.
3. Create `SerializeCapability [S0]` dynamically.
4. Reparent the same integer object/ID under the new S0 parent.
5. Replicate integer into transient `S0-C`.
6. Ask Qwen to specialize the copy into `FloatMultiplication`.
7. Verify the generated implementation.
8. Activate the verified result as `S1`.
9. Link float directly under SerializeCapability.

### Final hierarchy
```text
              SerializeCapability [S0]
                        │
               ┌────────┴────────┐
               ▼                 ▼
 IntegerMultiplication    FloatMultiplication
        [S0]                    [S1]
```

The original integer capability remains S0. The S0-C copy is transient and does not appear in the final hierarchy.

## 6. State model

| State | Meaning |
|---|---|
| `S0` | Original/general capability state |
| `S0-C` | Transient replicated copy used for specialization |
| `GENERATED` | Generated source before activation |
| `S1` | Verified and active specialized capability |
| `FAILED` | Generation or verification failed |

## 7. Supervisor checklist

- 🔵 Initial state: only `IntegerMultiplication [S0]`.
- 🟡 Float request: required typed capability is missing.
- 🧩 Serialization/generalization: `SerializeCapability [S0]` appears only now.
- 🔗 Reparent: original integer keeps its ID and remains S0.
- 🔁 Replication: transient `S0-C` copy is created.
- 🧠 Specialization: Ollama + `qwen2.5-coder:7b`.
- 🛡️ Verification gates activation.
- 🟢 Float becomes S1.
- 🌳 Final hierarchy has integer S0 and float S1 as siblings.
- 💾 Metadata/source are persisted.
- 🔄 Reload/reuse does not regenerate the float capability.

In [25]:
# ============================================================
# SPS-CA — ADD & MULTIPLY TEST
# ============================================================

from pathlib import Path
from sps_specialization import CapabilityRegistry

print("=" * 70)
print("SPS-CA: ADD & MULTIPLY TEST")
print("=" * 70)

# Load persisted registry
registry = CapabilityRegistry(
    storage_dir=Path("/tmp/sps-capability-registry")
)
registry.load()

# ------------------------------------------------------------
# 1. Show capabilities
# ------------------------------------------------------------
print("\nAvailable capabilities:")

capabilities = registry.all()

for cap in capabilities:
    print(f"  - {cap.name} [{cap.state}]")

# ------------------------------------------------------------
# 2. Integer Multiplication
# ------------------------------------------------------------
print("\n" + "-" * 70)
print("INTEGER MULTIPLICATION")
print("-" * 70)

integer_mul = registry.find("IntegerMultiplication")

if integer_mul:
    result = integer_mul.execute(6, 7)
    print(f"IntegerMultiplication [{integer_mul.state}]")
    print("6 × 7 =", result)

    assert result == 42
else:
    print("❌ IntegerMultiplication not found")


# ------------------------------------------------------------
# 3. Float Multiplication
# ------------------------------------------------------------
print("\n" + "-" * 70)
print("FLOAT MULTIPLICATION")
print("-" * 70)

float_mul = registry.find("FloatMultiplication")

if float_mul:
    result = float_mul.execute(2.5, 4.0)
    print(f"FloatMultiplication [{float_mul.state}]")
    print("2.5 × 4.0 =", result)

    assert result == 10.0
else:
    print("⚠️ FloatMultiplication not found")


# ------------------------------------------------------------
# 4. Addition sanity tests
# ------------------------------------------------------------
print("\n" + "-" * 70)
print("ADDITION")
print("-" * 70)

result_int = 10 + 5
result_float = 2.5 + 3.5

print("10 + 5     =", result_int)
print("2.5 + 3.5 =", result_float)

assert result_int == 15
assert result_float == 6.0


# ------------------------------------------------------------
# 5. Final verification
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("✅ ADD & MULTIPLY TEST PASSED")
print("=" * 70)

SPS-CA: ADD & MULTIPLY TEST

Available capabilities:
  - IntegerMultiplication [S0]
  - SerializeCapability [S0]
  - FloatMultiplication [S1]

----------------------------------------------------------------------
INTEGER MULTIPLICATION
----------------------------------------------------------------------
IntegerMultiplication [S0]
6 × 7 = 42

----------------------------------------------------------------------
FLOAT MULTIPLICATION
----------------------------------------------------------------------
FloatMultiplication [S1]
2.5 × 4.0 = 10.0

----------------------------------------------------------------------
ADDITION
----------------------------------------------------------------------
10 + 5     = 15
2.5 + 3.5 = 6.0

✅ ADD & MULTIPLY TEST PASSED


In [23]:
from IPython.display import display, HTML

# Prevent Jupyter from limiting large outputs
display(HTML("""
<style>
.jp-OutputArea-output {
    max-height: none !important;
    overflow: visible !important;
}
</style>
"""))